In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np
import json
import torch
import math

In [ ]:
file_path = "/kaggle/input/squid-dataset/train.json"
squad_dict = {}
with open(file_path,'r') as f:
  squad_dict = json.load(f)

In [ ]:
min_context_len = 10000;
max_context_len = 0;
average_context_size = 0;
total_number_of_context = 0;

In [ ]:
questionContext = {};
data = squad_dict['data']
row_Data = []  #creating list
context_index  = 0
cnt = 0
for article in data:
    for paragraph in article['paragraphs']:
        context = paragraph['context']

        context_words = context.split()
        min_context_len = min(min_context_len, len(context_words))
        max_context_len = max(max_context_len, len(context_words))
        total_number_of_context += 1

        for qa in paragraph['qas']:
            if not qa['is_impossible']:
                question = qa['question']
                for ans in qa['answers']:
                    answer = ans['text']
                    start = ans['answer_start']
                    end = start + len(answer)
                    row_Data.append(
                        {
                            'context':context,
                            'question':question,
                            'answer':answer,
                            'start':start,
                            'end':end,
                            'c_id':context_index
                        }
                    )
                    #adding question context
                    questionContext[question] = context_index
        context_index += 1

In [ ]:
print("min_context_len : ",min_context_len)
print("max_context_len : ",max_context_len)
print("total_number_of_context : ",total_number_of_context)

counter = 0
for i in questionContext:
  counter+=1;
print("size of question Context : ", counter)

In [ ]:
row = pd.DataFrame(row_Data)

In [ ]:
len(questionContext)

l = row['context'].tolist()
documents = row[['context','c_id']].drop_duplicates().reset_index(drop=True)
documents

In [ ]:
!pip install transformers faiss-cpu datasets

In [ ]:
import faiss
model_name = "facebook/contriever" #Bart  Robert based model # mini llm #
from transformers import AutoTokenizer, AutoModel
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

model.eval()

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

In [ ]:
import torch.nn.functional as F

def encode_text(texts):
    encoded_input = tokenizer(texts, return_tensors='pt', padding=True, truncation=True).to(device)
    with torch.no_grad():
        output = model(**encoded_input)
        # embeddings = output.last_hidden_state[:, 0]  # CLS token

        #adding new
        embeddings = output.last_hidden_state
        attention_mask = encoded_input['attention_mask'].unsqueeze(-1)
        masked_embeddings = embeddings * attention_mask
        sum_embeddings = masked_embeddings.sum(dim=1)
        lengths = attention_mask.sum(dim=1)
        mean_pooled = sum_embeddings / lengths
        embeddings = F.normalize(mean_pooled, p=2, dim=1)
    return embeddings.cpu().numpy()

def encode_in_batches(texts, batch_size=32):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        batch_embeddings = encode_text(batch_texts)
        all_embeddings.append(batch_embeddings)
    return np.vstack(all_embeddings)


In [ ]:
texts = documents['context'].tolist()
doc_embeddings = encode_in_batches(texts, batch_size=32)

print("doc_embedding is done !!!")

In [ ]:
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

In [ ]:
question = []
for keys, value in questionContext.items():
  question.append(keys)

In [ ]:
import math
import numpy as np

def dcg(scores):
    return sum([rel / math.log2(idx + 2) for idx, rel in enumerate(scores)])

def compute_metrics(question, questionContext, index, k=10, max_samples=2000):
    total_correct = 0
    reciprocal_ranks = []
    ndcg_scores = []
    precision_scores = []
    recall_scores = []
    mar_scores = []  # To hold MAR for each query

    for idx, ques in enumerate(question):
        if idx >= max_samples:
            break

        query_embedding = encode_text([ques])
        distances, indices = index.search(query_embedding, k)

        predictions = indices[0]
        actual_index = questionContext[ques]

        # Relevance: 1 if prediction is the correct one
        relevance = [1 if pred == actual_index else 0 for pred in predictions]

        # Accuracy / Recall@k
        if actual_index in predictions:
            total_correct += 1
            rank = list(predictions).index(actual_index)
            reciprocal_ranks.append(1.0 / (rank + 1))
            recall_scores.append(1.0)  # Only one relevant item exists
            precision_scores.append(1.0 / (rank + 1))  # Precision for that one item
        else:
            reciprocal_ranks.append(0.0)
            recall_scores.append(0.0)
            precision_scores.append(0.0)

        # NDCG
        ideal_relevance = sorted(relevance, reverse=True)
        ndcg = dcg(relevance) / dcg(ideal_relevance) if sum(ideal_relevance) > 0 else 0.0
        ndcg_scores.append(ndcg)

        # MAR: For each query, compute recall at k
        num_relevant_items = sum(relevance)  # The number of relevant items in the top-k
        mar_scores.append(num_relevant_items / len(relevance) if num_relevant_items > 0 else 0.0)

    num_samples = min(len(question), max_samples)

    return {
        f"Accuracy@{k}": (total_correct / num_samples) * 100,
        f"MRR@{k}": np.mean(reciprocal_ranks),
        f"NDCG@{k}": np.mean(ndcg_scores),
        f"Recall@{k}": np.mean(recall_scores),
        f"Precision@{k}": np.mean(precision_scores),
        f"MAR@{k}": np.mean(mar_scores)  # Adding MAR to the metrics
    }


In [ ]:
samples = [2000, 5000, 10000]

final_result = []

for spl in samples:
    temp = []
    for k in [1, 2, 3, 4, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100]:
        results = compute_metrics(
            question = question,
            questionContext=questionContext,
            index = index,
            k=k,
            max_samples=spl
        )
        temp.append(results)
        print(f"\n🔍 Top-{k} Evaluation Metrics:")
        for metric, value in results.items():
            print(f"{metric}: {value:.4f}")
    final_result.append(temp)

In [ ]:
temp = []
for k in [1, 2, 3, 4, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100]:
        results = compute_metrics(
            question = question,
            questionContext=questionContext,
            index = index,
            k=k,
            max_samples=20000
        )
        temp.append(results)
        print(f"\n🔍 Top-{k} Evaluation Metrics:")
        for metric, value in results.items():
            print(f"{metric}: {value:.4f}")
final_result.append(temp)

In [ ]:
temp = []
for k in [1, 2, 3, 4, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100]:
        results = compute_metrics(
            question = question,
            questionContext=questionContext,
            index = index,
            k=k,
            max_samples=86769
        )
        temp.append(results)
        print(f"\n🔍 Top-{k} Evaluation Metrics:")
        for metric, value in results.items():
            print(f"{metric}: {value:.4f}")
final_result.append(temp) 

In [ ]:
with open("/kaggle/working/final_result.json", "w") as f:
    json.dump(final_result, f)